In [2]:
HW_PATH = "MyDrive/AA 272 - GPS Class/HW4" # <-- CHANGE THIS PATH IF USING COLAB

%pip install pypdf -qq

import os
import re
import glob
import subprocess
import pypdf

# Mount Google Drive if using Colab
try:
    from google.colab import drive

    if not os.path.ismount("/content/drive"):
        print("Mounting Google Drive...")
        drive.mount("/content/drive")
    else:
        print("Google Drive already mounted")
    gdrive = True
    notebooks_dir = os.path.join("/content/drive", HW_PATH)
except:
    print("Not using Colab")
    # get current dir
    notebooks_dir = os.getcwd()
    print(f"Current directory: {notebooks_dir}")
    gdrive = False

# Install LaTeX packages if using Colab
if gdrive:
    print("Installing LaTeX packages...")
    !apt-get update -qq > /dev/null
    !apt-get install -y texlive-latex-base texlive-latex-recommended texlive-fonts-recommended texlive-xetex pandoc > /dev/null
    %pip install pypandoc -q > /dev/null


# Check notebooks directory exists
if not os.path.exists(notebooks_dir):
    raise ValueError(f"Error: Path {notebooks_dir} does not exist")

notebook_files = sorted(glob.glob(os.path.join(notebooks_dir, "AA272*.ipynb")))
if not notebook_files:
    raise ValueError(f"Warning: No notebook files found in {notebooks_dir}")

import shutil

template_zip_path = os.path.join(notebooks_dir, "latex_template.zip")
template_folder_path = os.path.join(notebooks_dir, "latex_template")

# If the template folder exists, copy it to ./
if os.path.exists(template_folder_path) and os.path.isdir(template_folder_path):
    dst_folder = "./latex_template"
    print(f"Copying {template_folder_path} to {dst_folder} ...")
    if os.path.exists(dst_folder):
        shutil.rmtree(dst_folder)
    shutil.copytree(template_folder_path, dst_folder)
# Otherwise, try extracting latex_template.zip if it exists
elif os.path.exists(template_zip_path):
    import zipfile
    print(f"Extracting {template_zip_path}...")
    with zipfile.ZipFile(template_zip_path, 'r') as zip_ref:
        zip_ref.extractall("./")
else:
    raise ValueError(f"Error: Neither {template_folder_path} nor {template_zip_path} found")

pdf_files = []
for i, notebook_file in enumerate(notebook_files):
    print(f"Processing {i+1}/{len(notebook_files)}: {notebook_file}")
    output_filename = os.path.basename(os.path.splitext(notebook_file)[0])
    output_dir = "./"

    command = [
        "jupyter",
        "nbconvert",
        "--to",
        "pdf",
        f"--template=./latex_template",
        "--output-dir",
        output_dir,
        "--output",
        output_filename,
        notebook_file,
    ]
    try:
        subprocess.run(command, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(f"\n[Error]: Failed to convert {notebook_file} to PDF.")
        print("Command:", " ".join(command))
        print("Return code:", e.returncode)
        print("Output:", e.output)
        print("Error Output:", e.stderr)
        continue
    pdf_files.append(os.path.join(output_dir, f"{output_filename}.pdf"))

# Combine all PDFs into one
pdf_files = sorted(pdf_files)
if pdf_files:
    print("Combining PDFs...")
    merger = pypdf.PdfWriter()
    for pdf_path in pdf_files:
        merger.append(pdf_path)
    output_filename = re.sub(r"_P\d+", "_Submission", os.path.basename(pdf_files[0]))

    # Save to current directory (./)
    output_path_current = os.path.join("./", output_filename)
    merger.write(output_path_current)

    # Save to notebooks directory
    output_path_notebooks = os.path.join(notebooks_dir, output_filename)
    merger.write(output_path_notebooks)

    merger.close()

    # Remove individual PDF files after combining
    for pdf_path in pdf_files:
        if os.path.exists(pdf_path):
            os.remove(pdf_path)
    print(f"Combined {len(pdf_files)} PDFs into '{output_filename}'")

Google Drive already mounted
Installing LaTeX packages...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Copying /content/drive/MyDrive/AA 272 - GPS Class/HW4/latex_template to ./latex_template ...
Processing 1/5: /content/drive/MyDrive/AA 272 - GPS Class/HW4/AA272_2025_HW4_P0.ipynb
Processing 2/5: /content/drive/MyDrive/AA 272 - GPS Class/HW4/AA272_2025_HW4_P1.ipynb
Processing 3/5: /content/drive/MyDrive/AA 272 - GPS Class/HW4/AA272_2025_HW4_P2.ipynb
Processing 4/5: /content/drive/MyDrive/AA 272 - GPS Class/HW4/AA272_2025_HW4_P3.ipynb
Processing 5/5: /content/drive/MyDrive/AA 272 - GPS Class/HW4/AA272_2025_HW4_SelfCare.ipynb
Combining PDFs...
Combined 5 PDFs into 'AA272_2025_HW4_Submission.pdf'
